# 03 LightGBM Modeling (Single Route)
## Context
- We now have a clean dataset and more understanding of the data distribution
## Goal
- At this stage, I will train a LightGBM model on a single route and two pre-sepcified stops for proof-of-concept
- Compare its performance with baseline random guessing
- Model on a longer route vs a shorter route to compare their performances

# Import

In [1]:
import polars as pl
import plotly.express as px
from constants import DATA_FILE, holidays, long_holidays, DEMO_FOLDER
from datetime import datetime, date
from helpers import (
    clean_df,
    create_travel_time_column,
    create_time_features,
    ml_data_preprocess,
)
import lightgbm as gbm
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

pl.Config.set_tbl_rows(30)

polars.config.Config

# Load Main Dataset

In [3]:
df_raw = pl.scan_parquet(DATA_FILE)
df_raw.head().collect()

PlateNumb,OperatorID,OperatorNo,RouteUID,RouteID,RouteNameZh_tw,RouteNameEn,SubRouteUID,SubRouteID,SubRouteNameZh_tw,SubRouteNameEn,Direction,StopUID,StopID,StopNameZh_tw,StopNameEn,StopSequence,MessageType,DutyStatus,BusStatus,A2EventType,GPSTime,TripStartTimeType,TripStartTime,TransTime,SrcRecTime,SrcTransTime,SrcUpdateTime,UpdateTime
str,i32,i32,str,i32,str,str,str,str,str,str,cat,str,i32,str,str,i32,cat,cat,cat,cat,str,cat,str,str,str,str,str,str
"""KKA-3975""",16,702,"""THB0968""",968,"""0968""","""0968""","""THB096802""","""096802""","""0968""","""0968""","""返程""","""THB300178""",300178,"""大竹消防隊""","""Dajhu Fire Brigade""",10,"""非定期""","""正常""","""正常""","""進站""","""2025-02-28T18:38:03+08:00""","""實際發車時間""","""2025-02-28T17:42:24+08:00""",null,"""2025-02-28T18:38:04+08:00""","""2025-02-28T18:38:04+08:00""",null,"""2025-02-28T18:38:06+08:00"""
"""396-FQ""",35,1308,"""THB0968""",968,"""0968""","""0968""","""THB096802""","""096802""","""0968""","""0968""","""返程""","""THB300198""",300198,"""庫倫街口""","""Kulun St. Entrance""",1,"""非定期""","""正常""","""正常""","""進站""","""2025-02-28T18:37:03+08:00""","""實際發車時間""","""2025-02-28T18:37:03+08:00""",null,"""2025-02-28T18:37:03+08:00""","""2025-02-28T18:37:03+08:00""",null,"""2025-02-28T18:37:06+08:00"""
"""396-FQ""",35,1308,"""THB0968""",968,"""0968""","""0968""","""THB096802""","""096802""","""0968""","""0968""","""返程""","""THB300198""",300198,"""庫倫街口""","""Kulun St. Entrance""",1,"""非定期""","""正常""","""正常""","""離站""","""2025-02-28T18:37:21+08:00""","""實際發車時間""","""2025-02-28T18:37:21+08:00""",null,"""2025-02-28T18:37:21+08:00""","""2025-02-28T18:37:21+08:00""",null,"""2025-02-28T18:37:26+08:00"""
"""396-FQ""",35,1308,"""THB0968""",968,"""0968""","""0968""","""THB096802""","""096802""","""0968""","""0968""","""返程""","""THB300197""",300197,"""長榮""","""Changrong""",2,"""非定期""","""正常""","""正常""","""進站""","""2025-02-28T16:01:00+08:00""","""實際發車時間""","""2025-02-28T15:38:50+08:00""",null,"""2025-02-28T16:01:00+08:00""","""2025-02-28T16:01:00+08:00""",null,"""2025-02-28T16:01:01+08:00"""
"""396-FQ""",35,1308,"""THB0968""",968,"""0968""","""0968""","""THB096802""","""096802""","""0968""","""0968""","""返程""","""THB300197""",300197,"""長榮""","""Changrong""",2,"""非定期""","""正常""","""正常""","""離站""","""2025-02-28T16:01:14+08:00""","""實際發車時間""","""2025-02-28T15:38:50+08:00""",null,"""2025-02-28T16:01:15+08:00""","""2025-02-28T16:01:15+08:00""",null,"""2025-02-28T16:01:16+08:00"""


# First Target Route (1728)

In [91]:
df = df_raw.filter(pl.col("RouteID") == 1728).pipe(clean_df).collect()

## Create features
- More EDA on the interaction effects between day of week and rush hour effects
- In src.helpers I wrote functions that generate time features such as is_morning_rush, is_evening_rush etc.

In [92]:
df_target = create_travel_time_column(df, "新竹轉運站", "龍潭運動公園", "返程").pipe(
    create_time_features
)
df_target.glimpse()
# 7037 rows

Rows: 7037
Columns: 19
$ Route                      <str> '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280', '17280'
$ Plate                      <str> '058-FS', 'KKB-2058', 'KKB-2035', 'KKB-1715', 'KKB-2056', '058-FS', 'KKB-1713', 'KKB-1692', 'KKB-2059', 'KKB-1715'
$ Direction                  <cat> 返程, 返程, 返程, 返程, 返程, 返程, 返程, 返程, 返程, 返程
$ Stop                       <str> '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站', '新竹轉運站'
$ StopSeq                    <i32> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
$ Event                      <cat> 離站, 離站, 離站, 離站, 離站, 離站, 離站, 離站, 離站, 離站
$ Time              <datetime[μs]> 2025-02-28 06:01:29, 2025-02-28 06:59:02, 2025-02-28 08:01:19, 2025-02-28 09:00:50, 2025-02-28 10:28:42, 2025-02-28 11:38:58, 2025-02-28 12:01:27, 2025-02-28 13:01:40, 2025-02-28 14:02:01, 2025-02-28 15:13:43
$ Day_of_week                <cat> Fri, Fri, Fri, Fri, Fri, Fri, Fri, Fri, Fri, Fri
$ DepartureTime     <datetime[μs]> 2025

In [93]:
df_target.head()

Route,Plate,Direction,Stop,StopSeq,Event,Time,Day_of_week,DepartureTime,Route_right,Direction_right,Stop_right,StopSeq_right,Event_right,Day_of_week_right,ArrivalTime,Duration_min,is_rush_hour,day_of_week
str,str,cat,str,i32,cat,datetime[μs],cat,datetime[μs],str,cat,str,i32,cat,cat,datetime[μs],f64,cat,cat
"""17280""","""058-FS""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:01:29,"""Fri""",2025-02-28 06:01:29,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 06:45:15,43.766667,"""not_rush""","""Fri"""
"""17280""","""KKB-2058""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 06:59:02,"""Fri""",2025-02-28 06:59:02,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 07:41:21,42.316667,"""not_rush""","""Fri"""
"""17280""","""KKB-2035""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 08:01:19,"""Fri""",2025-02-28 08:01:19,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 08:52:13,50.9,"""morning_rush""","""Fri"""
"""17280""","""KKB-1715""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 09:00:50,"""Fri""",2025-02-28 09:00:50,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 10:06:22,65.533333,"""morning_rush""","""Fri"""
"""17280""","""KKB-2056""","""返程""","""新竹轉運站""",1,"""離站""",2025-02-28 10:28:42,"""Fri""",2025-02-28 10:28:42,"""17280""","""返程""","""龍潭運動公園""",9,"""進站""","""Fri""",2025-02-28 11:40:29,71.783333,"""not_rush""","""Fri"""


## Interaction between day of week and rush hour

In [94]:
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

for day in day_order:
    px.box(
        df_target.filter(pl.col("day_of_week") == day),
        x="is_rush_hour",
        y="Duration_min",
        title=f"{day}",
    ).show()

- The weekdays are affected by rush hours
- Friday has especially strong rush hour 
- Weekends are not affected by rush hour and have more stable but longer travel time compared to weekdays

In [95]:
px.histogram(
    df_target.filter(
        pl.col("day_of_week").is_in(["Sat", "Sun"]),
        # pl.col("is_rush_hour") == "evening_rush",
    ),
    x="Duration_min",
    nbins=30,
)


- Long tail distribution of travel time
- Scaling consideration
    - It's hard to judge rush hours for 1000+ routes with vastly different itineraries and travel times
    - In general, LightGBM is amazing at binning continous features. Therefore is_rush_hour would not be incorporated into the model

# Modeling
1. Model choice: LightGBM
2. Consideration
    - Appropriate model for this datasize
    - When scaling, the `SubRouteID` feature would be categorical with 1000+ features, which LightGBM habdles natively.
    - More computationally efficient, which is critical for the datasize.
    - Extremely efficient dataset handling (`gbm.Dataset`)
    - Prediction accuracy is the primary consideration. Interpretability is not needed.
    - Model is lightweight and can be deployed more easily
3. Features
    1. Day of Week (Mon, Tue etc.)
    2. Time in a day (represented by minutes passed from midnight; therefore it's a continuous feature from 0 to 1440)
4. Evaluation
    - Since it's time series data, use 2025 Feb to Nov as training set.
    - Use 2025 Dec as evaluation set
    - Not using 2026 data (for now) since there is Chinese New Year there which might affect the general model evaluation

In [96]:
# Training set: 2025 Feb to November
# Eval set: 2025 December
# To avoid (for now) Chinese New Year on 2026 in the test set
df_train = (
    df_target.filter(
        # remove faulty data points
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 2, 25), datetime(2025, 11, 30, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)
df_test = (
    df_target.filter(
        # remove faulty data points
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        )
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "Duration_min",
        ]
    )
)

# only 1 point was removed

In [97]:
X_train = df_train.drop("Duration_min").to_pandas()
y_train = df_train.select("Duration_min").to_pandas()
X_test = df_test.drop("Duration_min").to_pandas()
y_test = df_test.select("Duration_min").to_pandas()

In [98]:
# use simple default settings for the trial
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [99]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000263 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 182
[LightGBM] [Info] Number of data points in the train set: 5386, number of used features: 2
[LightGBM] [Info] Start training from score 58.917335


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [100]:
y_pred = model.predict(X_test)

In [101]:
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [102]:
strict_criterion = y_train.mean().item() * 0.05
loose_criterion = y_train.mean().item() * 0.1

In [103]:
display(
    output.select(
        (abs(pl.col("prediction") - pl.col("label")) < loose_criterion)
        .mean()
        .alias("within_loose_criterion"),
        (abs(pl.col("prediction") - pl.col("label")) < strict_criterion)
        .mean()
        .alias("within_strict_criterion"),
    )
)
print(f"{(output['prediction'] - output['label']).abs().quantile(0.9):.2f}")
# ML metrics reporting
print(f"r_2 = {r2_score(output['label'], output['prediction']):.2f}")
print(f"RMSE = {root_mean_squared_error(output['label'], output['prediction']):.2f}")
print(f"MAE = {mean_absolute_error(output['label'], output['prediction']):.2f}")

within_loose_criterion,within_strict_criterion
f64,f64
0.684124,0.412439


11.40
r_2 = 0.55
RMSE = 7.20
MAE = 5.13


## Model Performance (simple features)
1. **Custom Metrics**
- MAE, RMSE etc do not reflect how the users (passengers) percieve a predictive model like this.
- I used 10% and 5% of mean travel time as a threshold to judge if a prediction is *accurate*
- For most passengers, on a route with mean travel time 56 minutes, a prediction off by 5.6 minutes is usually acceptable
- Scaling consideration
    - Needs to standardize the loose/strict threshold for most prediction intervals
2. **Performance**
- loose criterion: 68.41%
    - meaning out of 100 predictions, 68 are within +/- 5.6 minutes of the true travel time
- strict criterion: 41.24%
    - meaning out of 100 predictions, 41 are within +/- 2.8 minutes of the true travel time
3. **Comments**
- With extremely simple features, the model performed way better than my expectations.
- 41% of predictions are only off by 2.8 minutes (which is often negligible by bus passengers)
- I consider the model to be working really well by the thresholds

## Try Full Features
1. Day of week
2. Time
3. Holidays
4. Long holidays (連假)
- 3, 4 are based on my intuition that during holidays, the traffic on highways gets extremely heavy
- These four features are my original full features in mind

In [104]:
df_train = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 2, 25), datetime(2025, 11, 30, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        ),
        # create is_holiday, is_long_holiday
        pl.col("Time").dt.date().is_in(holidays).alias("is_holiday"),
        pl.col("Time").dt.date().is_in(long_holidays).alias("is_long_holiday"),
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "is_holiday",
            "is_long_holiday",
            "Duration_min",
        ]
    )
)

df_test = (
    df_target.filter(
        pl.col("Duration_min").is_between(20, 120),
        pl.col("Time").is_between(
            date(2025, 12, 1), datetime(2025, 12, 31, 23, 59, 59)
        ),
    )
    .with_columns(
        (pl.col("Time").dt.hour() * 60 + pl.col("Time").dt.minute()).alias(
            "minutes_from_midnight"
        ),
        pl.col("Time").dt.date().is_in(holidays).alias("is_holiday"),
        pl.col("Time").dt.date().is_in(long_holidays).alias("is_long_holiday"),
    )
    .select(
        [
            "minutes_from_midnight",
            "day_of_week",
            "is_holiday",
            "is_long_holiday",
            "Duration_min",
        ]
    )
)

In [105]:
X_train = df_train.drop("Duration_min").to_pandas()
y_train = df_train.select("Duration_min").to_pandas()
X_test = df_test.drop("Duration_min").to_pandas()
y_test = df_test.select("Duration_min").to_pandas()

In [106]:
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [107]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000199 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 186
[LightGBM] [Info] Number of data points in the train set: 5386, number of used features: 4
[LightGBM] [Info] Start training from score 58.917335


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [108]:
y_pred = model.predict(X_test)

In [109]:
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [110]:
output.select(
    (abs(pl.col("prediction") - pl.col("label")) < loose_criterion)
    .mean()
    .alias("within_loose_criterion"),
    (abs(pl.col("prediction") - pl.col("label")) < strict_criterion)
    .mean()
    .alias("within_strict_criterion"),
)

within_loose_criterion,within_strict_criterion
f64,f64
0.687398,0.414075


In [111]:
feat_importance = pl.DataFrame(
    {
        "feature": X_train.columns,
        "importance": model.booster_.feature_importance(importance_type="gain"),
    }
).sort("importance", descending=True)

feat_importance

feature,importance
str,f64
"""minutes_from_midnight""",3.3995e6
"""day_of_week""",346109.161107
"""is_holiday""",47599.306293
"""is_long_holiday""",2474.939041


## Model Performance (Full features)
- Incorporating the two features actually degraded the model slightly
- Potential reasons
    - Not enough data points for the holiday features 
    - Holidays might not be as strong a factor as I anticipated
    - day_of_week and time already explain a major part of the holiday influence
- holiday features are abandoned in the final features
- The LightGBM model seems to capture the rush hour extremely well that it'd likely outperform my manual picking of rush hour times

## Baseline Random Guessing
- Baseline guessing: naively guessing the mean travel time for every trip

In [112]:
# Baseline guessing
n = y_test.shape[0]

output = pl.DataFrame(
    {
        "guess_mean": [y_train.mean().item()] * n,
        "guess_median": [y_train.median().item()] * n,
        "label": y_test["Duration_min"],
    }
)

display(
    output.select(
        (abs(pl.col("guess_mean") - pl.col("label")) < loose_criterion)
        .mean()
        .alias("within_loose_criterion"),
        (abs(pl.col("guess_mean") - pl.col("label")) < strict_criterion)
        .mean()
        .alias("within_strict_criterion"),
    )
)

(output["guess_mean"] - output["label"]).abs().quantile(0.9)
# ML metrics reporting
print(f"r_2 = {r2_score(output['label'], output['guess_mean']):.2f}")
print(f"RMSE = {root_mean_squared_error(output['label'], output['guess_mean']):.2f}")
print(f"MAE = {mean_absolute_error(output['label'], output['guess_mean']):.2f}")

within_loose_criterion,within_strict_criterion
f64,f64
0.433715,0.234043


r_2 = -0.02
RMSE = 10.80
MAE = 8.29


## Baseline Performance
1. **Loose criterion**: 43.34%
2. **Strict criterion** 23.40%

## Conclusion
1. For strict criterion, my model improved the accuracy by 65%
2. For loose criterion, my model improved the accuracy by 43%
3. Meaning my model improves predictions **more** when judged with a stricter criterion, which makes the model more useful
4. The result shows that the model is learning real patterns from the data

# Second Target Route (7500G)
- Now try a longer route (3+ hours) to see how well the model works

In [119]:
df_7500 = (
    df_raw.filter(pl.col("SubRouteID").str.starts_with("7500G"))
    .pipe(clean_df)
    .collect()
)

## Fuzzy join tolerance
- Previously I manually set the tolerance range by checking the actual travel time between the stops
- We can more systematically gauge the optimal tolerance range by which value would render the most matches
- This would allow us to calculate the optimal tolerance values between thousands of stops

In [120]:
tolerance_range = [f"{i}h" for i in range(2, 10)]
for t in tolerance_range:
    print(
        f"{t}: {create_travel_time_column(df_7500, '臺北轉運站', '臺南轉運站', '返程', t).height}"
    )

2h: 5
3h: 9
4h: 1770
5h: 6190
6h: 6601
7h: 6627
8h: 6627
9h: 6627


- The matched results plateaued at 7h, which makes sense considering this is a longer route

In [121]:
df_target_7500 = create_travel_time_column(
    df_7500, "臺北轉運站", "臺南轉運站", "返程", "7h"
)
# I modularize the above ML dataset preprocess into src.helpers
X_train, X_test, y_train, y_test = ml_data_preprocess(
    df_target_7500, date(2025, 12, 31)
)

In [123]:
px.histogram(df_target_7500, x="Duration_min", title="Travel Duration of Route 7500G")

- There exist wrong matches as shown in the plot. The bus is physically impossible to travel that fast.
- Future consideration: ml_data_preprocess needs to be able to strip out anomaly travel times

In [124]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000209 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 260
[LightGBM] [Info] Number of data points in the train set: 5533, number of used features: 2
[LightGBM] [Info] Start training from score 258.363904


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [125]:
y_pred = model.predict(X_test)
output = pl.DataFrame({"prediction": y_pred, "label": y_test["Duration_min"]})

In [126]:
strict_criterion = y_train.mean().item() * 0.05
loose_criterion = y_train.mean().item() * 0.1

output.select(
    (abs(pl.col("prediction") - pl.col("label")) < loose_criterion)
    .mean()
    .alias("within_loose_criterion"),
    (abs(pl.col("prediction") - pl.col("label")) < strict_criterion)
    .mean()
    .alias("within_strict_criterion"),
)

within_loose_criterion,within_strict_criterion
f64,f64
0.902194,0.660878


In [127]:
# ML metrics reporting
print(f"r_2 = {r2_score(output['label'], output['prediction']):.2f}")
print(f"RMSE = {root_mean_squared_error(output['label'], output['prediction']):.2f}")
print(f"MAE = {mean_absolute_error(output['label'], output['prediction']):.2f}")

r_2 = 0.52
RMSE = 19.03
MAE = 12.65


## Model Performance
1. Loose criterion (25 minutes error tolerance): **90.22%**
2. Strict criterion (13 minutes error tolerance): **66.09%**
3. Using the same (10%, 5%) criteria, the accuracies are even higher for this longer route. However, since the actual tolerated errors are difference for different routes, they can't be directly compared.
4. For a route this long, 2 out of 3 guesses are off by less than 13 minutes also seem quite impressive and would be genuinely useful for real users.

# Another custom metric
- Since directly using (10%, 5%) as the error tolerance threshold might be misleading for different routes, here's a more natural metric
- A natural metric is *the time limit criterion that would lead to a 90% coverage of prediction accuracy*.
    - For example, this metric for route 1728 (the shorter route) is 11.4 minutes. This means **90%** of the time my model will predict a travel time that is off by **less than** *11.4 minutes*. 
    - This allows users to mentally estimate their travel time, which could encourage them to take inter-city busses more as they are now deemed *predictable*

# Baseline Random Guessing (7500G)

In [130]:
n = y_test.shape[0]

output = pl.DataFrame(
    {
        "guess_mean": [y_train.mean().item()] * n,
        "label": y_test["Duration_min"],
    }
)

display(
    output.select(
        (abs(pl.col("guess_mean") - pl.col("label")) < loose_criterion)
        .mean()
        .alias("within_loose_criterion"),
        (abs(pl.col("guess_mean") - pl.col("label")) < strict_criterion)
        .mean()
        .alias("within_strict_criterion"),
    )
)

within_loose_criterion,within_strict_criterion
f64,f64
0.691042,0.332724


In [129]:
# ML metrics reporting
print(f"r_2 = {r2_score(output['label'], output['guess_mean']):.2f}")
print(f"RMSE = {root_mean_squared_error(output['label'], output['guess_mean']):.2f}")
print(f"MAE = {mean_absolute_error(output['label'], output['guess_mean']):.2f}")

r_2 = -0.00
RMSE = 27.53
MAE = 21.50


## Baseline performance
1. Loose criterion: 69.10%
2. Strict criterion: 33.27%

## Conclusion
- Compared to the baseline guessing, my model (66.09%) improved the **strict criterion** accuracy by a whopping 98%
- When used on longer routes, my model seems to improve the baseline even more, which is a strong indicator on the usefulness of my model (since longer routes have more demands for prediction)
- From this single-route proof of concept model building,
    - The results are extremely promising
    - LightGBM with my simple feaures **CAN** capture meaningful patterns

# Model fitting for Demo website
- To conlcude phase 1 of the project (model building on single routes), I'm deploying a demo website.
- The model and features would be exactly the same one as above, but now trained on entire dataset.

In [4]:
df_demo = (
    df_raw.filter(pl.col("SubRouteID").str.starts_with("1728")).pipe(clean_df).collect()
)
df_target_demo = create_travel_time_column(
    df_demo, "新竹轉運站", "龍潭運動公園", "返程", "2h"
)

In [5]:
X_train, X_test, y_train, y_test = ml_data_preprocess(
    df_target_demo,
    date(
        2027, 12, 31
    ),  # separating on a future date since I'm now training on the full dataset
)

In [6]:
X_train["Day_of_week"].unique()

['Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu']
Categories (7, str): ['Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu']

- The categories are stored in src.constants.py for easier access

In [7]:
model = gbm.LGBMRegressor(
    num_leaves=31,
    learning_rate=0.05,
    min_child_samples=20,
    subsample=0.8,
    n_estimators=500,
    random_state=42,
)

In [8]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000154 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 217
[LightGBM] [Info] Number of data points in the train set: 7037, number of used features: 2
[LightGBM] [Info] Start training from score 58.958247


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [ ]:
# saving the model for deployment
model.booster_.save_model(DEMO_FOLDER / "demo_model_1728.txt")